In [1]:
import os
import sys
import pandas as pd

# 1. RISOLUZIONE DEI PERCORSI
# Questo trucco dice a Python di guardare anche nella cartella radice (food_recommender/)
# così puoi importare config.py e la cartella models senza errori di "ModuleNotFoundError"
sys.path.append(os.path.abspath(os.path.join('..')))

from config import PATH_CLEAN_RECIPES, PATH_CLEAN_INTERACTIONS
from models.hybrid_based import HybridRecommender

print("Moduli importati con successo!")

# 2. CARICAMENTO DATI
# Usiamo os.path.join('..', ...) perché il notebook si trova dentro la cartella 'notebooks/'
# e deve fare un passo indietro per trovare il dataset
df_recipes = pd.read_csv(os.path.join('..', PATH_CLEAN_RECIPES))
df_interactions = pd.read_csv(os.path.join('..', PATH_CLEAN_INTERACTIONS))

print(f"Ricette caricate: {len(df_recipes)}")
print(f"Interazioni caricate: {len(df_interactions)}")

Moduli importati con successo!
Ricette caricate: 15144
Interazioni caricate: 385801


In [ ]:
# 1. FIX IMPORT ED INIZIALIZZAZIONE DI TUTTI I MODELLI
from models.popularity import PopularityRecommender
from models.content_based import ContentBasedRecommender
from models.health_based import HealthBasedRecommender
from models.mood_based import MoodBasedRecommender
from models.collaborative_filtering import CollaborativeFilteringRecommender
from models.hybrid_based import HybridRecommender  # <-- Corretto da hybrid_based a hybrid

# Addestriamo i 5 pilastri (se li hai già eseguiti sopra, basta verificare che i nomi coincidano)
pop_model = PopularityRecommender(m=50)
pop_model.fit(df_recipes, df_interactions)

content_model = ContentBasedRecommender(matrix_path="../dataset/tfidf_matrix.npz")
content_model.fit(df_recipes)

health_model = HealthBasedRecommender()
health_model.fit(df_recipes)

mood_model = MoodBasedRecommender()
mood_model.fit(df_recipes)

cf_model = CollaborativeFilteringRecommender(min_user_interactions=5)
cf_model.fit(df_recipes, df_interactions)

# 2. ORA LANCERAI IL MODELLO IBRIDO SENZA ERRORI
hybrid_model = HybridRecommender(pop_model, content_model, mood_model, cf_model, health_model)

# 3. PREPARAZIONE INPUT PER ABLATION STUDY
USER_TEST = df_interactions['user_id'].value_counts().index[0] 
INGREDIENTS_TEST = ["chicken breast", "garlic"]
MOOD_TEST = {'body': 4.0, 'time': -3.0, 'taste': 4.0}
HEALTH_TEST = {'profile_name': 'weight_loss', 'max_calories': 600}

print("\n!!! Tutti i modelli sono pronti in memoria e l'ibrido è istanziato !!!")

-> Caricamento matrice TF-IDF pre-calcolata da ../dataset/tfidf_matrix.npz...
-> Estrazione automatica delle 6 dimensioni del Mood per ogni ricetta...
-> Matrice Mood generata e normalizzata in scala [-5, +5]!
-> Filtro Cold Start applicato. Righe rimanenti: 365262
-> Split Temporale completato. Train size: 351077, Test size: 14185
-> Addestramento dell'algoritmo SVD definitivo...
-> Modello SVD addestrato con successo!

!!! Tutti i modelli sono pronti in memoria e l'ibrido è istanziato !!!
